Gradio와 Document Intelligence 연동

In [ ]:
import gradio as gr
import requests
import time
import random
import platform
from PIL import Image, ImageDraw, ImageFont
import os
from dotenv import load_dotenv
from pathlib import Path

env_path = Path(os.getcwd()).parent / ".env"
load_dotenv(env_path)

endpoint = os.getenv("DOCUMENT_INTELLIGENCE_ENDPOINT")
key = os.getenv("DOCUMENT_INTELLIGENCE_KEY")

# --- 기능 1: API 호출 ---
def request_document_intelligence(image_path):
    full_url = f"{endpoint.rstrip('/')}/documentintelligence/documentModels/prebuilt-read:analyze?_overload=analyzeDocument&api-version=2024-11-30"
    headers = {
        "Ocp-Apim-Subscription-Key": key,
        "Content-Type": "image/png"
    }
    with open(image_path, "rb") as image_file:
        image_data = image_file.read()

    response = requests.post(full_url, headers=headers, data=image_data)
    if response.status_code != 202:
        return None

    url = response.headers['Operation-Location']
    while True:
        result_response = requests.get(url, headers=headers)
        result_response_json = result_response.json()
        current_status = result_response_json.get("status")
        if current_status == "running":
            time.sleep(1)
            continue
        else:
            break
    return result_response_json if current_status == "succeeded" else None

# --- 기능 2: 이미지 그리기 ---
def draw_image(image_path, response_data):
    image = Image.open(image_path)
    draw = ImageDraw.Draw(image)
    block_list = response_data['analyzeResult']['paragraphs']
    
    for block in block_list:
        line_color = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))
        font = get_font()
        polygon = block['boundingRegions'][0]['polygon']
        content = block['content']
        polygon_point_list = [(polygon[i], polygon[i+1]) for i in range(0, len(polygon), 2)]
        draw.polygon(polygon_point_list, outline=line_color, width=2)
        draw.text((polygon[0], polygon[1] - 20), content, fill=line_color, font=font)
    return image

def get_font():
    font_size = 20
    try:
        if platform.system() == "Windows": return ImageFont.truetype("malgun.ttf", font_size)
        elif platform.system() == "Darwin": return ImageFont.truetype("AppleGothic.ttf", font_size)
        else: return ImageFont.load_default()
    except: return ImageFont.load_default()

# --- 기능 3: 메인 연동 로직 ---
def change_image(origin_image):
    if origin_image is None: return None
    response_data = request_document_intelligence(origin_image)
    if response_data:
        return draw_image(origin_image, response_data)
    return origin_image

# --- 기능 4: Gradio UI ---
with gr.Blocks() as demo:
    with gr.Row():
        input_image = gr.Image(label="이미지 선택", type="filepath", width=500)
        output_image = gr.Image(label="결과 이미지", type="pil", interactive=False, width=500)
    
    input_image.change(change_image, inputs=[input_image], outputs=[output_image])

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


c:\Users\EL045\AppData\Local\Programs\Python\Python314\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\EL045\AppData\Local\Programs\Python\Python314\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\EL045\AppData\Local\Programs\Python\Python314\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\EL045\AppData\Local\Programs\Python\Python314\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. 